In [1]:
import socket

In [2]:
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.connect(('192.168.1.68', 4000))

In [19]:
s.close()

In [3]:
"""
Command Name:
Communication Check
Description: Can be used to test the communication link, returns a single byte
Send Bytes: 3, 2, 1, 28
Return Byte: 0
"""
s.send(bytearray([3,2,1,28]))
resp = s.recv(1)
print(resp)


b'\x00'


In [4]:
s.send(bytearray([3,2,1,27]))
resp = s.recv(1024*1024)

ver_len = resp[0]
ver = resp[1:ver_len+1].decode()
print(f"{ver=}")

ser_len = resp[ver_len+1]
serial_num = resp[ver_len+2:ver_len+2+ser_len]
print(f"{serial_num=}")


print((resp))
for b in resp:
    print(repr(b))

ver='3.38'
serial_num=b'240407'
b'\x043.38\x06240407'
4
51
46
51
56
6
50
52
48
52
48
55


In [5]:
"""
Command Name:
Set Acq. Time
Description: Sets the data acquisition time, in seconds
Send Bytes: 3, 2, 1, 18, b0, b1, b2, b3 (where b0, b1, b2, b3 are the bytes for the single precision floating point value of
the data acquisition time, in little endian byte order)
Return Byte: 0 (indicates success)
"""

import struct

tacq = -5 # seconds

data_bytes = struct.pack('<f', tacq) # little endian (<) single precison float (f)
print(repr(data_bytes))
import binascii
print(binascii.b2a_hex(data_bytes,'-'))

s.send(bytearray([3,2,1,18])+data_bytes)
resp = s.recv(1024*1024)
print(resp)


b'\x00\x00\xa0\xc0'
b'00-00-a0-c0'
b'\x00'


In [7]:
s.send(bytearray([3,2,1,44]))
resp = s.recv(1024*1024)
print(resp)


b'I\x03\x00\x00Film_Sense_Data\r\n8\t1\t-72.331\t0.003\t-0.017\t3.8138\r\n367.01\t14.19\t2.75\t1.82\t1.197\t0.963\t0.095\r\n449.22\t15.78\t7.51\t5.93\t1.467\t1.723\t-0.049\r\n525.03\t23.60\t6.29\t3.57\t1.363\t1.229\t-0.169\r\n593.73\t13.82\t6.97\t3.88\t1.207\t1.014\t-0.186\r\n655.93\t17.64\t7.29\t3.76\t1.270\t0.986\t-0.282\r\n730.90\t25.07\t6.90\t3.38\t1.286\t1.170\t-0.223\r\n850.30\t23.40\t5.45\t4.62\t1.036\t0.947\t-0.275\r\n946.26\t25.15\t3.48\t6.28\t0.704\t0.950\t-0.269\r\n-0.05800\t0.40969\t0.91038\t1.01233\t2.1766\r\n0.32985\t0.43180\t0.83949\t1.02455\t4.1028\r\n0.49688\t0.36337\t0.78808\t1.01687\t2.5405\r\n0.59753\t0.30354\t0.74217\t1.00712\t1.2165\r\n0.66361\t0.25737\t0.70241\t1.00109\t3.1069\r\n0.72518\t0.21017\t0.65570\t0.99668\t3.0764\r\n0.79505\t0.15444\t0.58655\t0.99321\t3.3331\r\n0.83426\t0.12495\t0.53702\t0.99012\t2.2714\r\n3\r\nFit_Diff\t0.0244\r\nThick(nm).3\t37.242\r\nAngle\t73.827\r\nModel: Alumina on si test1, Date: 2024-11-21 11:48, Ver: 3.38, Temp: 25.2'


In [9]:
bytes(resp[:4])

b'I\x03\x00\x00'

In [11]:
struct.unpack('<I', resp[:4])

(841,)

In [12]:
len(resp)

845

In [15]:
s.send(bytearray([3,2,1,44]))
resp = s.recv(4)
print(resp)
data_len = struct.unpack('<I', resp[:4])[0]
resp = s.recv(data_len)
print(resp)
with open("test.txt", 'wb') as f:
    f.write(resp)


b'H\x03\x00\x00'
b'Film_Sense_Data\r\n8\t1\t-72.331\t0.003\t0.009\t3.7918\r\n367.01\t14.19\t2.75\t1.82\t1.197\t0.963\t0.095\r\n449.22\t15.78\t7.51\t5.93\t1.467\t1.723\t-0.049\r\n525.03\t23.60\t6.29\t3.57\t1.363\t1.229\t-0.169\r\n593.73\t13.82\t6.97\t3.88\t1.207\t1.014\t-0.186\r\n655.93\t17.64\t7.29\t3.76\t1.270\t0.986\t-0.282\r\n730.90\t25.07\t6.90\t3.38\t1.286\t1.170\t-0.223\r\n850.30\t23.40\t5.45\t4.62\t1.036\t0.947\t-0.275\r\n946.26\t25.15\t3.48\t6.28\t0.704\t0.950\t-0.269\r\n-0.04241\t0.39716\t0.91677\t1.01016\t2.2471\r\n0.33878\t0.42899\t0.83737\t1.02411\t4.0137\r\n0.50328\t0.36088\t0.78516\t1.01653\t2.4996\r\n0.60314\t0.30139\t0.73850\t1.00746\t1.2194\r\n0.66859\t0.25531\t0.69843\t1.00155\t3.0634\r\n0.72969\t0.20860\t0.65118\t0.99691\t3.0438\r\n0.79878\t0.15351\t0.58171\t0.99352\t3.2960\r\n0.83758\t0.12451\t0.53194\t0.99070\t2.2699\r\n3\r\nFit_Diff\t0.0224\r\nThick(nm).3\t36.976\r\nAngle\t73.814\r\nModel: Alumina on si test1, Date: 2024-11-21 11:53, Ver: 3.38, Temp: 25.4'


In [18]:
s.send(bytearray([3,2,1,32,4]))
resp = s.recv(4)
print(resp)
data_len = struct.unpack('<I', resp[:4])[0]
resp = s.recv(data_len)
print(resp)
with open("test_model.txt", 'wb') as f:
    f.write(resp)


b'I\x03\x00\x00'
b'Film_Sense_Model\r\nComment Line\r\n2\t0\t0\t0\t4\t0\t0\t0\t5\t0\t0.000000\t0.000000\t0.000000\t0.000000\t0\t0\t1.332000\t0.002200\t0\t0\t0\t0\t0.050000\t633.000000\t0\t1\t1\t1.000000\t0\t100.000000\t0\t0\t0\t0\t0\t0.000000\t0\t0.000000\t0\r\nSi (Herzinger)\r\n1\t0\t0\t0.000000\t0\t0.000000\t100.000000\t0.000000\t0\t0.000000\t0.000000\t0.000000\t0.000000\t0\t"(none)"\r\n0\t72.331000\t64.000000\t66.000000\t0.000000\r\nSiO2 (Native, Herzinger)\r\n1\t0\t0\t0.000000\t0\t0.000000\t100.000000\t0.000000\t0\t0.000000\t0.000000\t0.000000\t0.000000\t0\t"(none)"\r\n0\t1.000000\t0.000000\t0.000000\t0.000000\r\nCauchy\r\n2\t4\t0\t0.000000\t0\t0.000000\t100.000000\t0.000000\t0\t0.000000\t0.000000\t0.000000\t0.000000\t0\t"(none)"\r\n1\t22.410000\t20.000000\t30.000000\t10.000000\r\n1\t1.777000\t0.000000\t6.000000\t-1.000000\r\n1\t0.000000\t0.000000\t0.200000\t-1.000000\r\n0\t0.000000\t0.000000\t0.000000\t0.000000\r\n0\t0.000000\t0.000000\t0.000000\t0.000000\r\n\t\t\t\t\t\t\t\t\t\t\t